# Synapse vs Databricks Data Validation

Reusable notebook that compares tables between Synapse and Databricks.
Define your table mappings below, then run all cells to get a full comparison report.

**Requirements:** `pyodbc`, `polars`, `databricks-sql-connector`

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, os.path.abspath('..'))

import pyodbc
from databricks import sql
import polars as pl

from validation_utils import compare_synapse_vs_databricks

## Configuration

Edit the table mappings and connection details below.

In [ ]:
# ─── Table Mappings ─────────────────────────────────────────────────────────────
# Each entry: (synapse_schema, synapse_table, databricks_catalog, databricks_schema, databricks_table)

TABLE_MAPPINGS = [
    # (synapse_schema, synapse_table, dbx_catalog, dbx_schema, dbx_table)
    ("hstg_v", "vw_hstg_UDLMMRSIOC_mmrs_cs_vw_cycle", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_cycle"),
    ("hstg_v", "vw_hstg_UDLMMRSIOC_mmrs_cs_vw_event", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_event"),
    ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_location", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_location"),
    ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_rf_material", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_material"),
    ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_rf_timecode", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_timecode"),
    ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_rf_unit", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_unit"),
    ("hstg_v", "vw_hstg_UDLMMRSIOC_vw_unit_fleet_fleettype", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_unit_fleet_fleettype"),
]

In [ ]:
# ─── Connection Setup ──────────────────────────────────────────────────────────

# Synapse
synapse_server = 'syn-rt-prd-unity.sql.azuresynapse.net'
synapse_database = 'syndwrtprdunity'
synapse_username = 'byambadorjme@riotinto.com'
synapse_driver = '{ODBC Driver 17 for SQL Server}'

synapse_connection_string = f"""
    DRIVER={synapse_driver};
    SERVER={synapse_server};
    DATABASE={synapse_database};
    UID={synapse_username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""

# Databricks
databricks_access_token = os.environ["DATABRICKS_TOKEN"]
databricks_server_hostname = 'adb-2439498039309203.3.azuredatabricks.net'
databricks_http_path = '/sql/1.0/warehouses/e7d134c4348ac8b5'

# ─── Connect ──────────────────────────────────────────────────────────────────
print("Connecting to Synapse (browser auth prompt may appear)...")
synapse_conn = pyodbc.connect(synapse_connection_string)
print("✓ Synapse connected")

databricks_conn = sql.connect(
    server_hostname=databricks_server_hostname,
    http_path=databricks_http_path,
    access_token=databricks_access_token,
)
print("✓ Databricks connected")

In [ ]:
# ─── Display settings ─────────────────────────────────────────────────────────
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(50)
pl.Config.set_fmt_str_lengths(100)

## Run Comparison

Iterates over all table mappings and produces a summary report.

In [ ]:
# ─── Run all comparisons ──────────────────────────────────────────────────────
all_results = []

for i, (syn_schema, syn_table, dbx_catalog, dbx_schema, dbx_table) in enumerate(TABLE_MAPPINGS, 1):
    print(f"\n{'═' * 80}")
    print(f"  [{i}/{len(TABLE_MAPPINGS)}] {syn_schema}.{syn_table} → {dbx_catalog}.{dbx_schema}.{dbx_table}")
    print(f"{'═' * 80}")

    try:
        result = compare_synapse_vs_databricks(
            databricks_conn=databricks_conn,
            synapse_conn=synapse_conn,
            synapse_schema_name=syn_schema,
            synapse_table_name=syn_table,
            databricks_catalog_name=dbx_catalog,
            databricks_schema_name=dbx_schema,
            databricks_table_name=dbx_table,
        )
        result["synapse_table"] = syn_table
        result["databricks_table"] = dbx_table
        all_results.append(result)

        print(f"  Schema Match: {result['schema_matches']}")
        print(f"  Row Count Match: {result['row_count_matches']}")
        print(f"  Synapse: {result['synapse_row_count']:,} | Databricks: {result['databricks_row_count']:,}")
        if result['sampling_applied']:
            print(f"  ** Sampled (last 30 days): Syn={result['sampled_row_count_synapse']:,} | Dbx={result['sampled_row_count_databricks']:,}")
        print(f"  Data Match: {result['matched_rows']:,}/{result['syn_total']:,} ({result['pct_of_synapse']:.2f}%)")
        print(f"  Hash Match: {result['full_match']}")

    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        all_results.append({
            "synapse_table": syn_table,
            "databricks_table": dbx_table,
            "error": str(e),
        })

print(f"\n{'═' * 80}")
print(f"  DONE: {len(all_results)} tables compared")
print(f"{'═' * 80}")

In [ ]:
# ─── Summary Report ───────────────────────────────────────────────────────────
summary_rows = []
for r in all_results:
    if "error" in r:
        summary_rows.append({
            "table": r["databricks_table"],
            "synapse_rows": None,
            "databricks_rows": None,
            "schema_match": None,
            "row_count_match": None,
            "data_match_pct": None,
            "hash_match": None,
            "status": f"ERROR: {r['error'][:50]}",
        })
    else:
        status = "✓ Perfect" if r["full_match"] else (
            "⚠ Row count mismatch" if not r["row_count_matches"] else "⚠ Data mismatch"
        )
        summary_rows.append({
            "table": r["databricks_table"],
            "synapse_rows": r["synapse_row_count"],
            "databricks_rows": r["databricks_row_count"],
            "schema_match": r["schema_matches"],
            "row_count_match": r["row_count_matches"],
            "data_match_pct": round(r["pct_of_synapse"], 2),
            "hash_match": r["full_match"],
            "status": status,
        })

summary_df = pl.DataFrame(summary_rows)
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)
print(summary_df)

In [ ]:
# ─── Detailed Results (per table) ─────────────────────────────────────────────
# Uncomment a table index to inspect its detailed results

# idx = 0  # Change index to inspect a specific table
# r = all_results[idx]
# print(f"Table: {r['databricks_table']}")
# print(f"\nSchema Comparison:")
# display(r['schema_comparison_df'])
# print(f"\nNumeric Profile:")
# display(r['numeric_profile_comparison_df'])
# print(f"\nString Profile:")
# display(r['string_profile_comparison_df'])

In [ ]:
# ─── Cleanup ──────────────────────────────────────────────────────────────────
synapse_conn.close()
databricks_conn.close()
print("Connections closed.")